# 7N0C Interface Discovery Analysis

**Date:** February 7, 2026  
**Status:** Week 3, Day 4  
**Target:** NSP10-NSP14 Exonuclease Complex  
**Approach:** DISCOVERY-BASED + COMPARISON with 6W4H

---

## Objectives
1. ✓ Load 7N0C structure
2. ✓ Identify NSP10 and NSP14 chains
3. 🔍 DISCOVER NSP10-NSP14 hot spots
4. 🔍 Map complete interface
5. 📊 COMPARE NSP10 binding: 7N0C (NSP10-NSP14) vs 6W4H (NSP10-NSP16)
6. ✓ Define docking grid box
7. ✓ Export results

---

## Key Questions
1. **How does NSP10 bind to NSP14 vs NSP16?**
   - Are the hot spots conserved?
   - Same binding mode or different?
   - K76 (NSP10) important in both?

2. **NSP14 interface druggability?**
   - Strong hot spots?
   - Suitable for small molecules?

---

## Section 1: Setup and Imports

In [2]:
# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

# Import libraries
import py3Dmol
from Bio import PDB
import numpy as np
import pandas as pd
import json
import os
from pathlib import Path
from datetime import datetime

# Configure paths
PDB_DIR = Path('../data/structures/pdb')
RESULTS_DIR = Path('../data/analysis_results')
PDB_FILE = PDB_DIR / '7N0C.pdb'

# Verify setup
RESULTS_DIR.mkdir(exist_ok=True, parents=True)
assert PDB_FILE.exists(), f"PDB file not found: {PDB_FILE}"

print("✓ Setup complete")
print(f"  PDB file: {PDB_FILE.resolve()}")
print(f"  Results: {RESULTS_DIR.resolve()}")
print(f"  File exists: {PDB_FILE.exists()}")
print()
print("📊 COMPARISON MODE: Will compare NSP10 binding with 6W4H")

✓ Setup complete
  PDB file: /Users/user/Desktop/Botanique/Project/data/structures/pdb/7N0C.pdb
  Results: /Users/user/Desktop/Botanique/Project/data/analysis_results
  File exists: True

📊 COMPARISON MODE: Will compare NSP10 binding with 6W4H


## Section 2: Load and Parse Structure

In [3]:
# Parse structure
parser = PDB.PDBParser(QUIET=True)
structure = parser.get_structure('7N0C', str(PDB_FILE))
model = structure[0]

print("="*70)
print("STRUCTURE INFORMATION - 7N0C")
print("="*70)
print()

# First pass - collect all chain info
all_chains = []
for chain in model:
    residues = [r for r in chain.get_residues() if r.id[0] == ' ']
    num_res = len(residues)
    
    if residues:
        first_res = residues[0].id[1]
        last_res = residues[-1].id[1]
    else:
        first_res = last_res = 'N/A'
    
    all_chains.append({
        'id': chain.id,
        'num_res': num_res,
        'first': first_res,
        'last': last_res
    })
    
    print(f"Chain {chain.id}: {num_res} residues (PDB {first_res}-{last_res})")

print()
print("Analyzing chain sizes...")
print()

# Chain detection for NSP10-NSP14
chain_assignments = {}
for chain_data in all_chains:
    num_res = chain_data['num_res']
    chain_id = chain_data['id']
    
    # NSP10-NSP14 specific detection
    if 125 < num_res < 145:
        protein = 'NSP10 (cofactor)'
        chain_assignments['nsp10'] = chain_id
    elif 500 < num_res < 530:
        protein = 'NSP14 (exonuclease)'
        chain_assignments['nsp14'] = chain_id
    elif num_res < 30:
        protein = 'RNA/Substrate/Ligand'
    else:
        protein = 'Unknown'
    
    print(f"Chain {chain_id} ({num_res} res) → {protein}")

print()
print("="*70)
print("FINAL ASSIGNMENTS:")
print("="*70)
if 'nsp10' in chain_assignments:
    print(f"  NSP10 = Chain {chain_assignments['nsp10']}")
else:
    print("  NSP10 = NOT FOUND ❌")
    
if 'nsp14' in chain_assignments:
    print(f"  NSP14 = Chain {chain_assignments['nsp14']}")
else:
    print("  NSP14 = NOT FOUND ❌")

print()
print("Target Interface for Discovery:")
print("  NSP10-NSP14 exonuclease complex")
print()
print("📊 COMPARISON: Will compare with 6W4H (NSP10-NSP16)")
print("="*70)

STRUCTURE INFORMATION - 7N0C

Chain A: 131 residues (PDB 1-131)
Chain B: 513 residues (PDB 1-523)
Chain T: 25 residues (PDB 5-29)
Chain D: 24 residues (PDB 48-71)

Analyzing chain sizes...

Chain A (131 res) → NSP10 (cofactor)
Chain B (513 res) → NSP14 (exonuclease)
Chain T (25 res) → RNA/Substrate/Ligand
Chain D (24 res) → RNA/Substrate/Ligand

FINAL ASSIGNMENTS:
  NSP10 = Chain A
  NSP14 = Chain B

Target Interface for Discovery:
  NSP10-NSP14 exonuclease complex

📊 COMPARISON: Will compare with 6W4H (NSP10-NSP16)


## Section 3: Visualize Structure (py3Dmol)

In [4]:
# Read PDB file
with open(PDB_FILE, 'r') as f:
    pdb_data = f.read()

# Create viewer
view = py3Dmol.view(width=800, height=600)
view.addModel(pdb_data, 'pdb')

# Color by chain
if 'nsp10' in chain_assignments:
    view.setStyle({'chain': chain_assignments['nsp10']}, {'cartoon': {'color': 'green', 'opacity': 0.8}})
if 'nsp14' in chain_assignments:
    view.setStyle({'chain': chain_assignments['nsp14']}, {'cartoon': {'color': 'orange', 'opacity': 0.8}})

# Hide small chains (RNA/ligands)
for chain_data in all_chains:
    if chain_data['num_res'] < 30:
        view.setStyle({'chain': chain_data['id']}, {'cartoon': {'opacity': 0.2, 'color': 'gray'}})

# Add labels
if 'nsp10' in chain_assignments:
    nsp10_chain = model[chain_assignments['nsp10']]
    nsp10_residues = [r for r in nsp10_chain.get_residues() if r.id[0] == ' ']
    if len(nsp10_residues) > 10:
        mid_res = nsp10_residues[len(nsp10_residues)//2].id[1]
        view.addLabel(f"NSP10 (Chain {chain_assignments['nsp10']})", 
                      {'fontColor': 'white', 'fontSize': 14, 'backgroundColor': 'green', 'backgroundOpacity': 0.8},
                      {'chain': chain_assignments['nsp10'], 'resi': mid_res})

if 'nsp14' in chain_assignments:
    nsp14_chain = model[chain_assignments['nsp14']]
    nsp14_residues = [r for r in nsp14_chain.get_residues() if r.id[0] == ' ']
    if len(nsp14_residues) > 10:
        mid_res = nsp14_residues[len(nsp14_residues)//2].id[1]
        view.addLabel(f"NSP14 (Chain {chain_assignments['nsp14']})", 
                      {'fontColor': 'white', 'fontSize': 14, 'backgroundColor': 'orange', 'backgroundOpacity': 0.8},
                      {'chain': chain_assignments['nsp14'], 'resi': mid_res})

view.zoomTo()
view.setBackgroundColor('white')

print("3D Structure - NSP10-NSP14 Exonuclease Complex:")
if 'nsp10' in chain_assignments:
    print(f"  Green = NSP10 (Chain {chain_assignments['nsp10']})")
if 'nsp14' in chain_assignments:
    print(f"  Orange = NSP14 (Chain {chain_assignments['nsp14']})")
print(f"  Gray (transparent) = RNA/ligands")

view.show()

3D Structure - NSP10-NSP14 Exonuclease Complex:
  Green = NSP10 (Chain A)
  Orange = NSP14 (Chain B)
  Gray (transparent) = RNA/ligands


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## Section 4: DISCOVER NSP10-NSP14 Hot Spots

In [5]:
print("="*70)
print("DISCOVERING HOT SPOTS: NSP10-NSP14 INTERFACE")
print("="*70)
print()

if 'nsp10' not in chain_assignments or 'nsp14' not in chain_assignments:
    print("⚠️  Missing required chains!")
    df_hot_10_14 = pd.DataFrame()
else:
    nsp10 = model[chain_assignments['nsp10']]
    nsp14 = model[chain_assignments['nsp14']]
    
    # Get all atoms
    nsp10_atoms = [atom for res in nsp10 if res.id[0] == ' ' for atom in res.get_atoms()]
    nsp14_atoms = [atom for res in nsp14 if res.id[0] == ' ' for atom in res.get_atoms()]
    
    print(f"NSP10 (Chain {chain_assignments['nsp10']}): {len(nsp10_atoms)} atoms")
    print(f"NSP14 (Chain {chain_assignments['nsp14']}): {len(nsp14_atoms)} atoms")
    print()
    print("Searching for ALL charged interactions (< 5 Å)...")
    print("This may take a minute...")
    print()
    
    # Find ALL interface contacts < 5 Å
    interface_contacts = []
    for atom1 in nsp10_atoms:
        for atom2 in nsp14_atoms:
            dist = np.linalg.norm(atom1.get_coord() - atom2.get_coord())
            if dist < 5.0:
                res1 = atom1.get_parent()
                res2 = atom2.get_parent()
                interface_contacts.append((res1, res2, dist, atom1.get_name(), atom2.get_name()))
    
    print(f"✓ Found {len(interface_contacts)} atomic contacts < 5 Å")
    print()
    
    # Get unique residue pairs
    unique_pairs = {}
    for res1, res2, dist, atom1, atom2 in interface_contacts:
        key = (res1.id[1], res2.id[1])
        if key not in unique_pairs or dist < unique_pairs[key][2]:
            unique_pairs[key] = (res1, res2, dist, atom1, atom2)
    
    print(f"✓ Unique residue pairs: {len(unique_pairs)}")
    print()
    
    # DISCOVER ALL charged interactions
    print("="*70)
    print("CHARGED INTERACTION DISCOVERY")
    print("="*70)
    print()
    
    charged_positive = ['LYS', 'ARG']
    charged_negative = ['ASP', 'GLU']
    
    hot_spots_10_14 = []
    for (pdb1, pdb2), (res1, res2, dist, atom1, atom2) in unique_pairs.items():
        resname1 = res1.get_resname()
        resname2 = res2.get_resname()
        
        # Check for charged interactions
        if (resname1 in charged_positive and resname2 in charged_negative) or \
           (resname1 in charged_negative and resname2 in charged_positive):
            
            interaction_type = 'Salt bridge' if dist < 4.0 else 'Ionic interaction'
            
            hot_spots_10_14.append({
                'NSP10_Res': resname1,
                'NSP10_PDB': pdb1,
                'NSP14_Res': resname2,
                'NSP14_PDB': pdb2,
                'Distance': dist,
                'Atom1': atom1,
                'Atom2': atom2,
                'Type': interaction_type
            })
    
    if hot_spots_10_14:
        df_hot_10_14 = pd.DataFrame(hot_spots_10_14).sort_values('Distance')
        
        print(f"✓ DISCOVERED {len(df_hot_10_14)} charged interactions!")
        print()
        print("All Charged Interactions (sorted by distance):")
        print()
        print(df_hot_10_14.to_string(index=False))
        print()
        
        # Highlight strongest interaction
        strongest = df_hot_10_14.iloc[0]
        print("="*70)
        print("🔥 STRONGEST HOT SPOT (NSP10-NSP14):")
        print(f"  {strongest['NSP10_Res']}{strongest['NSP10_PDB']} (NSP10) ←→ {strongest['NSP14_Res']}{strongest['NSP14_PDB']} (NSP14)")
        print(f"  Distance: {strongest['Distance']:.2f} Å")
        print(f"  Type: {strongest['Type']}")
        print("="*70)
        print()
        
        # Check for K76 (key residue from 6W4H)
        print("🔍 CHECKING FOR K76 (key residue from 6W4H NSP10-NSP16):")
        k76_found = df_hot_10_14[df_hot_10_14['NSP10_PDB'] == 76]
        if not k76_found.empty:
            print("  ✅ K76 IS INVOLVED in NSP10-NSP14 interface!")
            print(f"  Partner: {k76_found.iloc[0]['NSP14_Res']}{k76_found.iloc[0]['NSP14_PDB']}")
            print(f"  Distance: {k76_found.iloc[0]['Distance']:.2f} Å")
            print("  → K76 appears to be a conserved NSP10 hot spot!")
        else:
            print("  ⚠️  K76 NOT found in charged interactions")
            print("  → NSP10 may use different binding mode with NSP14")
        print("="*70)
    else:
        print("⚠️  No charged interactions found < 5 Å")
        df_hot_10_14 = pd.DataFrame()

print()

DISCOVERING HOT SPOTS: NSP10-NSP14 INTERFACE

NSP10 (Chain A): 955 atoms
NSP14 (Chain B): 4090 atoms

Searching for ALL charged interactions (< 5 Å)...
This may take a minute...

✓ Found 1354 atomic contacts < 5 Å

✓ Unique residue pairs: 175

CHARGED INTERACTION DISCOVERY

✓ DISCOVERED 1 charged interactions!

All Charged Interactions (sorted by distance):

NSP10_Res  NSP10_PDB NSP14_Res  NSP14_PDB  Distance Atom1 Atom2        Type
      LYS         93       ASP        126  2.784193    NZ   OD2 Salt bridge

🔥 STRONGEST HOT SPOT (NSP10-NSP14):
  LYS93 (NSP10) ←→ ASP126 (NSP14)
  Distance: 2.78 Å
  Type: Salt bridge

🔍 CHECKING FOR K76 (key residue from 6W4H NSP10-NSP16):
  ⚠️  K76 NOT found in charged interactions
  → NSP10 may use different binding mode with NSP14



## Section 5: Visualize NSP10-NSP14 Hot Spot

In [6]:
if not df_hot_10_14.empty:
    top_spot = df_hot_10_14.iloc[0]

    # --- REQUIRED columns ---
    required_cols = ['NSP10_PDB', 'NSP14_PDB', 'Distance',
                     'NSP10_Res', 'NSP14_Res']

    missing = [c for c in required_cols if c not in top_spot.index]
    if missing:
        raise ValueError(f"Missing columns in df_hot_10_14: {missing}")

    # --- FIXED ---
    nsp10_resi = int(top_spot['NSP10_PDB'])
    nsp14_resi = int(top_spot['NSP14_PDB'])
    distance  = float(top_spot['Distance'])

    print("Visualizing NSP10–NSP14 strongest hot spot...")
    print(
        f"Hot spot: "
        f"{top_spot['NSP10_Res']}{nsp10_resi} - "
        f"{top_spot['NSP14_Res']}{nsp14_resi}"
    )
    print(f"Distance: {distance:.2f} Å\n")

    with open(PDB_FILE, 'r') as f:
        pdb_data = f.read()

    view2 = py3Dmol.view(width=800, height=600)
    view2.addModel(pdb_data, 'pdb')

    # Proteins
    view2.setStyle(
        {'chain': chain_assignments['nsp10']},
        {'cartoon': {'color': 'green', 'opacity': 0.5}}
    )
    view2.setStyle(
        {'chain': chain_assignments['nsp14']},
        {'cartoon': {'color': 'orange', 'opacity': 0.5}}
    )

    # Hot spots
    view2.addStyle(
        {'chain': chain_assignments['nsp10'], 'resi': nsp10_resi},
        {'sphere': {'color': 'red', 'radius': 2.0}}
    )
    view2.addStyle(
        {'chain': chain_assignments['nsp14'], 'resi': nsp14_resi},
        {'sphere': {'color': 'blue', 'radius': 2.0}}
    )

    # Labels
    view2.addLabel(
        f"{top_spot['NSP10_Res']}{nsp10_resi}\n{distance:.1f} Å",
        {
            'fontColor': 'red',
            'fontSize': 14,
            'backgroundColor': 'white',
            'backgroundOpacity': 0.8
        },
        {'chain': chain_assignments['nsp10'], 'resi': nsp10_resi}
    )

    view2.addLabel(
        f"{top_spot['NSP14_Res']}{nsp14_resi}",
        {
            'fontColor': 'blue',
            'fontSize': 14,
            'backgroundColor': 'white',
            'backgroundOpacity': 0.8
        },
        {'chain': chain_assignments['nsp14'], 'resi': nsp14_resi}
    )

    view2.zoomTo({'chain': chain_assignments['nsp10'], 'resi': nsp10_resi})
    view2.setBackgroundColor('white')
    view2.show()

else:
    print("⚠️  No hot spots to visualize for NSP10–NSP14 interface")


Visualizing NSP10–NSP14 strongest hot spot...
Hot spot: LYS93 - ASP126
Distance: 2.78 Å



3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [8]:
if not df_hot_10_14.empty:
    top_spot = df_hot_10_14.iloc[0]

    # --- FIXED: correct residue names ---
    nsp10_resi = int(top_spot['NSP10_PDB'])
    nsp14_resi = int(top_spot['NSP14_PDB'])
    distance   = float(top_spot['Distance'])

    print("Visualizing NSP10–NSP14 strongest hot spot...")
    print(f"Hot spot: {top_spot['NSP10_Res']}{nsp10_resi} - "
          f"{top_spot['NSP14_Res']}{nsp14_resi}")
    print(f"Distance: {distance:.2f} Å\n")

    with open(PDB_FILE, 'r') as f:
        pdb_data = f.read()

    view2 = py3Dmol.view(width=800, height=600)
    view2.addModel(pdb_data, 'pdb')

    # Proteins
    view2.setStyle(
        {'chain': chain_assignments['nsp10']},
        {'cartoon': {'color': 'green', 'opacity': 0.5}}
    )
    view2.setStyle(
        {'chain': chain_assignments['nsp14']},
        {'cartoon': {'color': 'orange', 'opacity': 0.5}}
    )

    # Hot spot residues
    view2.addStyle(
        {'chain': chain_assignments['nsp10'], 'resi': nsp10_resi},
        {'sphere': {'color': 'red', 'radius': 2.0}}
    )
    view2.addStyle(
        {'chain': chain_assignments['nsp14'], 'resi': nsp14_resi},
        {'sphere': {'color': 'blue', 'radius': 2.0}}
    )

    # Labels
    view2.addLabel(
        f"{top_spot['NSP10_Res']}{nsp10_resi}\n{distance:.1f} Å",
        {'fontColor': 'red', 'fontSize': 14,
         'backgroundColor': 'white', 'backgroundOpacity': 0.8},
        {'chain': chain_assignments['nsp10'], 'resi': nsp10_resi}
    )

    view2.addLabel(
        f"{top_spot['NSP14_Res']}{nsp14_resi}",
        {'fontColor': 'blue', 'fontSize': 14,
         'backgroundColor': 'white', 'backgroundOpacity': 0.8},
        {'chain': chain_assignments['nsp14'], 'resi': nsp14_resi}
    )

    view2.zoomTo(
        {'chain': chain_assignments['nsp10'], 'resi': nsp10_resi}
    )
    view2.setBackgroundColor('white')
    view2.show()

else:
    print("⚠️  No hot spots to visualize for NSP10–NSP14 interface")


Visualizing NSP10–NSP14 strongest hot spot...
Hot spot: LYS93 - ASP126
Distance: 2.78 Å



3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## Section 6: Map NSP10-NSP14 Complete Interface (10 Å)

In [ ]:
print("="*70)
print("MAPPING NSP10-NSP14 INTERFACE (10 Å cutoff)")
print("="*70)
print()

if not df_hot_10_14.empty and 'nsp10' in chain_assignments and 'nsp14' in chain_assignments:
    # Get hot spot center
    top_spot = df_hot_10_14.iloc[0]
    res_10 = model[chain_assignments['nsp10']][top_spot['NSP10_PDB']]
    center = res_10['CA'].get_coord()
    
    print(f"Center point: NSP10 {top_spot['NSP10_Res']}{top_spot['NSP10_PDB']}")
    print(f"Coordinates: ({center[0]:.1f}, {center[1]:.1f}, {center[2]:.1f})")
    print()
    print("Finding all residues within 10 Å...")
    print()
    
    interface_10_14_full = []
    for chain in [model[chain_assignments['nsp10']], model[chain_assignments['nsp14']]]:
        for res in chain:
            if res.id[0] == ' ' and 'CA' in res:
                ca = res['CA'].get_coord()
                dist = np.linalg.norm(ca - center)
                
                if dist <= 10.0:
                    protein = 'NSP10' if chain.id == chain_assignments['nsp10'] else 'NSP14'
                    interface_10_14_full.append({
                        'Protein': protein,
                        'Chain': chain.id,
                        'Residue': res.get_resname(),
                        'PDB_Num': res.id[1],
                        'Distance': dist
                    })
    
    df_interface_10_14 = pd.DataFrame(interface_10_14_full).sort_values('Distance').reset_index(drop=True)
    
    print(f"✓ Found {len(df_interface_10_14)} interface residues:")
    print()
    print(df_interface_10_14.to_string(index=False))
    print()
    
    nsp10_count = len(df_interface_10_14[df_interface_10_14['Protein'] == 'NSP10'])
    nsp14_count = len(df_interface_10_14[df_interface_10_14['Protein'] == 'NSP14'])
    
    print("Interface composition:")
    print(f"  NSP10: {nsp10_count} residues")
    print(f"  NSP14: {nsp14_count} residues")
    print(f"  Total: {len(df_interface_10_14)} residues")
    print()
    print("📊 COMPARISON with 6W4H:")
    print("  6W4H (NSP10-NSP16): 22 residues (16 NSP10, 6 NSP16)")
    print(f"  7N0C (NSP10-NSP14): {len(df_interface_10_14)} residues ({nsp10_count} NSP10, {nsp14_count} NSP14)")
else:
    print("⚠️  Cannot map interface - missing hot spots or chains")
    df_interface_10_14 = pd.DataFrame()
    nsp10_count = nsp14_count = 0

print()
print("="*70)

## Section 7: Define Docking Grid Box


In [ ]:
from datetime import datetime
import pandas as pd

print('='*70)
print('DOCKING GRID BOX PARAMETERS')
print('='*70)

grid_box = {}

if 'df_hot_10_14' in globals() and not df_hot_10_14.empty:
    top = df_hot_10_14.iloc[0]
    res = model[chain_assignments['nsp10']][top['NSP10_PDB']]
    center = res['CA'].get_coord()

    grid_box = {
        'center_x': float(center[0]),
        'center_y': float(center[1]),
        'center_z': float(center[2]),
        'size_x': 25.0,
        'size_y': 25.0,
        'size_z': 25.0,
        'hot_spot': f"{top['NSP10_Res']}{top['NSP10_PDB']}-{top['NSP14_Res']}{top['NSP14_PDB']}"
    }

    for k, v in grid_box.items():
        print(f"{k}: {v}")
else:
    print('⚠️ No hot spots available — grid box not defined')


## Section 8: Summary & 6W4H Comparison


In [ ]:
print('='*70)
print('SUMMARY')
print('='*70)

if 'df_hot_10_14' in globals() and not df_hot_10_14.empty:
    top = df_hot_10_14.iloc[0]
    print(f"Top hot spot: NSP10 {top['NSP10_Res']}{top['NSP10_PDB']} ↔ NSP14 {top['NSP14_Res']}{top['NSP14_PDB']}")
    print(f"Distance: {top['Distance']:.2f} Å")
    print(f"Type: {top['Type']}")
else:
    print('No hot spots detected')


## Section 9: Export Results


In [ ]:
from pathlib import Path

RESULTS_DIR = Path('data/analysis_results')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

if 'df_hot_10_14' in globals() and not df_hot_10_14.empty:
    df_hot_10_14.to_csv(RESULTS_DIR / '7N0C_hotspots.csv', index=False)
    print('✓ Hot spots exported')

if 'df_interface_10_14' in globals() and not df_interface_10_14.empty:
    df_interface_10_14.to_csv(RESULTS_DIR / '7N0C_interface.csv', index=False)
    print('✓ Interface exported')


## Section 10: Conclusions

NSP10 shows conserved binding behavior if **K76** participates in both NSP14 and NSP16 interfaces.

**Implication:** Targeting the K76 region may disrupt multiple replication complexes.

✅ Analysis complete.